In [ ]:
from pathlib import Path
import sys


sys.path.append(str(Path().resolve().parent))

from utilities.src import (build_df, analyze_segment_lengths,
                           compare_sensor_means
                           

                                                            
                                                            )







In [154]:
BASE_DIR = Path().resolve()
PROJECT_ROOT = BASE_DIR.parent   #to go un up the the root
DATA_DIR = PROJECT_ROOT / "data"

# Root directory of dataset
root = Path(DATA_DIR)


Templates are useful because they provide a canonical representation of each movement.

They allow you to:

- understand the signal structure
- determine which sensor axes are informative
- design feature extraction
- build distance-based baselines

In other words, templates help you explore the dataset before building models.

In [155]:
templates_df=build_df(root)

value_cols = [
    'acc_x', 'acc_y', 'acc_z',
    'gyr_x', 'gyr_y', 'gyr_z',
    'mag_x', 'mag_y', 'mag_z'
]

merged_df = (
    templates_df
    .pivot_table(
        index=['subject', 'exercise', 'time index', 'execution_type'],
        columns='unit',
        values=value_cols,
        aggfunc='first'
    )
)

merged_df.columns = [f'{feature}_{unit}' for feature, unit in merged_df.columns]
merged_df = merged_df.reset_index()

merged_df.head(400)

,subject,exercise,time index,execution_type,acc_x_u1,acc_x_u2,acc_x_u3,acc_x_u4,acc_x_u5,acc_y_u1,...,mag_y_u1,mag_y_u2,mag_y_u3,mag_y_u4,mag_y_u5,mag_z_u1,mag_z_u2,mag_z_u3,mag_z_u4,mag_z_u5
0,s1,e1,314,correct,-9.675384,-9.623113,-3.064462,9.092290,-4.256507,-1.665096,...,0.457257,-0.485758,-0.794563,0.001059,0.733553,-0.081429,0.005508,-0.097777,-0.546160,0.216855
1,s1,e1,315,correct,-9.660426,-9.622773,-3.064653,9.077757,-4.254075,-1.687629,...,0.457905,-0.489491,-0.794913,0.001092,0.732285,-0.081475,0.005818,-0.096094,-0.547180,0.215713
2,s1,e1,316,correct,-9.660451,-9.681846,-3.049775,9.057949,-4.251647,-1.702569,...,0.457524,-0.493425,-0.794522,0.000979,0.732651,-0.081558,0.004581,-0.096575,-0.547338,0.216238
3,s1,e1,317,correct,-9.660356,-9.554703,-3.064489,9.163527,-4.249214,-1.657827,...,0.457243,-0.497159,-0.795803,0.002048,0.732930,-0.080656,0.004653,-0.096933,-0.548436,0.216331
4,s1,e1,318,correct,-9.645445,-9.404443,-3.086412,9.094555,-4.261387,-1.687700,...,0.457522,-0.501722,-0.795687,0.000966,0.731635,-0.081974,0.002505,-0.097586,-0.547134,0.216355
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,s1,e1,1763,low_amplitude,-9.674096,-3.976809,-2.836253,9.008036,-4.082684,-1.711807,...,0.460281,-0.761741,-0.776608,0.000092,0.727966,-0.060211,-0.144549,-0.168259,-0.561794,0.225208
396,s1,e1,1764,low_amplitude,-9.651635,-3.962023,-2.851306,9.003154,-4.099767,-1.726906,...,0.460677,-0.762129,-0.776747,-0.000417,0.727157,-0.059810,-0.144307,-0.168576,-0.562324,0.224760
397,s1,e1,1765,low_amplitude,-9.636693,-3.984255,-2.829126,9.015474,-4.087547,-1.734343,...,0.461334,-0.760817,-0.778479,0.001927,0.727473,-0.061476,-0.144115,-0.166657,-0.561786,0.225934
398,s1,e1,1766,low_amplitude,-9.636718,-3.932355,-2.836288,9.042505,-4.077818,-1.749283,...,0.459767,-0.761217,-0.777405,0.001157,0.727726,-0.061386,-0.143588,-0.167953,-0.562296,0.226316


In [156]:
filtered = merged_df.query(
    "exercise == 'e1' and execution_type == 'correct' and subject == 's1'"
)

filtered 

,subject,exercise,time index,execution_type,acc_x_u1,acc_x_u2,acc_x_u3,acc_x_u4,acc_x_u5,acc_y_u1,...,mag_y_u1,mag_y_u2,mag_y_u3,mag_y_u4,mag_y_u5,mag_z_u1,mag_z_u2,mag_z_u3,mag_z_u4,mag_z_u5
0,s1,e1,314,correct,-9.675384,-9.623113,-3.064462,9.092290,-4.256507,-1.665096,...,0.457257,-0.485758,-0.794563,0.001059,0.733553,-0.081429,0.005508,-0.097777,-0.546160,0.216855
1,s1,e1,315,correct,-9.660426,-9.622773,-3.064653,9.077757,-4.254075,-1.687629,...,0.457905,-0.489491,-0.794913,0.001092,0.732285,-0.081475,0.005818,-0.096094,-0.547180,0.215713
2,s1,e1,316,correct,-9.660451,-9.681846,-3.049775,9.057949,-4.251647,-1.702569,...,0.457524,-0.493425,-0.794522,0.000979,0.732651,-0.081558,0.004581,-0.096575,-0.547338,0.216238
3,s1,e1,317,correct,-9.660356,-9.554703,-3.064489,9.163527,-4.249214,-1.657827,...,0.457243,-0.497159,-0.795803,0.002048,0.732930,-0.080656,0.004653,-0.096933,-0.548436,0.216331
4,s1,e1,318,correct,-9.645445,-9.404443,-3.086412,9.094555,-4.261387,-1.687700,...,0.457522,-0.501722,-0.795687,0.000966,0.731635,-0.081974,0.002505,-0.097586,-0.547134,0.216355
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210,s1,e1,524,correct,-9.668614,-9.786056,-3.036743,9.073394,-4.154370,-1.584040,...,0.460933,-0.476475,-0.792575,0.006905,0.729776,-0.072331,-0.001532,-0.104783,-0.561114,0.221834
211,s1,e1,525,correct,-9.646166,-9.623375,-3.028275,9.053854,-4.154433,-1.614131,...,0.459607,-0.467283,-0.791764,0.006788,0.731131,-0.071240,0.001822,-0.104329,-0.562153,0.222982
212,s1,e1,526,correct,-9.646292,-9.548785,-2.968351,9.010007,-4.168982,-1.688832,...,0.459744,-0.460318,-0.791558,0.005202,0.731106,-0.071226,0.005768,-0.101855,-0.560987,0.223893
213,s1,e1,527,correct,-9.683885,-9.356911,-2.961959,9.063626,-4.144593,-1.763303,...,0.459101,-0.455840,-0.791062,0.007282,0.730575,-0.071708,0.006308,-0.097462,-0.561099,0.224673


# to do


we have 2 configuration wof sensors depending on leg or arm exercises, an idea would be to add a label to differentiate them since the sensors importance is different, morever one exercise is weird meaning that it only the torso moving and im  not sure about the sensors placemnent, we should insect it and see which one is the correct placement 

e.g column named type: 1 for leg, 0 for arm 

# Step 1 — inspect templates

In [157]:
lengths, summary, per_exercise = analyze_segment_lengths(merged_df)


sample_x = templates_df[templates_df["sample_id"] == 301]
print(len(sample_x))                          
print(sample_x["execution_type"].iloc[0])    
print(sample_x["time index"].min(), "→", sample_x["time index"].max())



results = compare_sensor_means(templates_df)

results[("e2", "u5")]

=== Timesteps per execution type ===
                 mean   std  min  max  count
execution_type                              
correct         142.3  30.9   98  218     40
fast             71.8  17.4   49  116     40
low_amplitude   125.2  27.5   84  203     40

=== Mean, min and max timesteps per exercise × execution type ===
          correct_max  correct_mean  correct_min  fast_max  fast_mean  \
exercise                                                                
e1                215         170.4          146       113       84.2   
e2                200         158.0          137        91       79.6   
e3                168         143.6          121       113       80.6   
e4                159         128.2           98        65       58.8   
e5                148         127.2           98        75       62.0   
e6                218         142.6          114       116       78.8   
e7                176         136.0           99        73       65.6   
e8            

{'acc':                                  acc_x   acc_y   acc_z
 execution_type                                        
 correct                        -3.2656 -8.6821 -0.2792
 low_amplitude                  -2.5998 -8.8514 -0.1379
 diff (low_amplitude - correct)  0.6658 -0.1693  0.1413,
 'gyr':                                  gyr_x   gyr_y   gyr_z
 execution_type                                        
 correct                         0.0042 -0.0012 -0.0098
 low_amplitude                   0.0011 -0.0040 -0.0082
 diff (low_amplitude - correct) -0.0031 -0.0028  0.0016,
 'mag':                                  mag_x   mag_y   mag_z
 execution_type                                        
 correct                        -0.1051  0.7114  0.1058
 low_amplitude                  -0.1626  0.7010  0.0831
 diff (low_amplitude - correct) -0.0575 -0.0104 -0.0227}

In [158]:
def flag_differences(results, negligible=0.02, noticeable=0.05):
    """
    Takes the output of compare_sensor_means and flags axes where
    the diff between execution types is noticeable.
 
    Uses a relative threshold: diff% = |diff| / |correct_mean| × 100
    so the comparison is fair across sensors with different scales.
 
    Thresholds (as fractions):
        < negligible          → negligible (noise)
        negligible – noticeable → marginal
        > noticeable          → noticeable
 
    Prints a summary and returns a DataFrame of flagged axes only.
    """
 
    rows = []
 
    for (exercise, unit), sensor_dict in results.items():
        for sensor, df in sensor_dict.items():
 
            # diff row is always the last row
            diff_row = df.iloc[-1]
            correct_row = df.loc["correct"] if "correct" in df.index else df.iloc[0]
 
            for col in diff_row.index:
                abs_diff = abs(diff_row[col])
                base = abs(correct_row[col])
 
                # avoid division by zero
                rel_diff = (abs_diff / base) if base > 1e-6 else 0.0
 
                if rel_diff < negligible:
                    flag = "negligible"
                elif rel_diff < noticeable:
                    flag = "marginal"
                else:
                    flag = "NOTICEABLE"
 
                rows.append({
                    "exercise": exercise,
                    "unit": unit,
                    "sensor": sensor,
                    "axis": col,
                    "correct_mean": round(correct_row[col], 4),
                    "diff": round(diff_row[col], 4),
                    "diff_%": round(rel_diff * 100, 2),
                    "flag": flag,
                })
 
    flagged_df = pd.DataFrame(rows)
 
    # Summary count
    counts = flagged_df["flag"].value_counts()
    print("=== Flag summary ===")
    print(counts.to_string())
    print()
 
    # Print only noticeable ones
    noticeable_df = flagged_df[flagged_df["flag"] == "NOTICEABLE"].sort_values(
        ["exercise", "unit", "sensor"]
    )
    print(f"=== NOTICEABLE differences (>{noticeable*100:.0f}% relative diff) ===")
    if noticeable_df.empty:
        print("None found.")
    else:
        print(noticeable_df.to_string(index=False))
 
    return flagged_df

#### On average, how much does each sensor axis vary during a specific exercise and execution type?

In [159]:
stats = merged_df.groupby(
    ['exercise', 'execution_type', 'subject']
).agg([ 'std']).reset_index()

stats.columns = ['_'.join(col).strip() for col in stats.columns]
stats=stats.drop(columns=['time index_std'])

In [160]:
sensor_avg = (
    stats
    .drop(columns=['subject_'])
    .groupby(['exercise_', 'execution_type_'])
    .mean()
    .reset_index()
)

In [161]:
sensor_avg.head(5)

,exercise_,execution_type_,acc_x_u1_std,acc_x_u2_std,acc_x_u3_std,acc_x_u4_std,acc_x_u5_std,acc_y_u1_std,acc_y_u2_std,acc_y_u3_std,...,mag_y_u1_std,mag_y_u2_std,mag_y_u3_std,mag_y_u4_std,mag_y_u5_std,mag_z_u1_std,mag_z_u2_std,mag_z_u3_std,mag_z_u4_std,mag_z_u5_std
0,e1,correct,0.053250,4.087398,0.712970,0.096087,0.105389,0.066535,3.120729,0.302042,...,0.002613,0.076546,0.029454,0.014724,0.002614,0.005204,0.039140,0.050850,0.016076,0.010460
1,e1,fast,0.070602,4.154153,0.760086,0.122475,0.120280,0.130962,3.315763,0.323637,...,0.006650,0.098321,0.030636,0.011700,0.003469,0.005409,0.037873,0.050408,0.013818,0.011245
2,e1,low_amplitude,0.030080,2.545773,0.396384,0.046874,0.049293,0.059501,2.644097,0.177716,...,0.001901,0.093270,0.016021,0.003832,0.001439,0.003408,0.028730,0.052219,0.006616,0.003530
3,e2,correct,0.089287,0.060314,0.213480,0.940298,0.317385,0.186179,0.138554,0.073806,...,0.009728,0.008060,0.010049,0.041783,0.006998,0.008611,0.007050,0.013618,0.264308,0.016471
4,e2,fast,0.126539,0.094467,0.238772,1.134879,0.417857,0.212769,0.209194,0.091866,...,0.012652,0.012582,0.013658,0.049614,0.010091,0.008896,0.010464,0.021980,0.284550,0.021492


In [166]:
limb_map = {
    "e1": "leg",
    "e2": "arm",
    "e3": "leg",
    "e4": "leg",
    "e5": "leg",
    "e6": "arm",
    "e7": "arm",
    "e8": "arm"
}

sensor_avg["limb"] = sensor_avg["exercise_"].map(limb_map)

#### For each exercise, which sensor unit has the highest variability?

In [167]:
import pandas as pd

df = pd.DataFrame()

In [168]:
# select columns by sensor type
acc_cols = [c for c in sensor_avg.columns if c.startswith("acc")]
gyr_cols = [c for c in sensor_avg.columns if c.startswith("gyr")]
mag_cols = [c for c in sensor_avg.columns if c.startswith("mag")]

# compute max per row for each sensor type
df["max_acc"] = sensor_avg[acc_cols].max(axis=1)
df["max_gyr"] = sensor_avg[gyr_cols].max(axis=1)
df["max_mag"] = sensor_avg[mag_cols].max(axis=1)
# find which column produced the maximum value
df["max_acc_sensor"] = sensor_avg[acc_cols].idxmax(axis=1)
df["max_gyr_sensor"] = sensor_avg[gyr_cols].idxmax(axis=1)
df["max_mag_sensor"] = sensor_avg[mag_cols].idxmax(axis=1)

df['exercise']=sensor_avg['exercise_']
df['type']=sensor_avg['execution_type_']
df['limb']=sensor_avg['limb']